In [6]:
split the train and test
danach kann ich schauen über plot_create dataset ob train and test die selbe block verteilung haben
falls ja alles perfekt und ich kann den RF mit den parametern von 
vorhin testen ob ich wieder auf die erwarteten selben werte komme
dann normalisieren und schauen ob die werte die selben bleiben mit dem RF classificator
dann knoten B bis H erstellen.

SyntaxError: invalid syntax (990101085.py, line 1)

In [38]:
# Cell 1: Split original datasets into train/test per node
import pandas as pd
import numpy as np
import os

# Configuration
nodes = ["A", "B", "C", "D", "E", "F", "G", "H"]
base_input_dir = "dataset"
output_dir = "dataset/normalized"

block_size = 1689
train_ratio = 0.7
#train_ratio = 0.99 # for the dataset with 100% synflood or nothing
random_state = 42

np.random.seed(random_state)

# Ensure output directory exists
os.makedirs(output_dir, exist_ok=True)

for node in nodes:
    # Build input and output paths
    input_path = os.path.join(base_input_dir, f"Node_{node}_final_synthetic_dataset_with_source.csv")
    
    print(f"Processing {input_path} ...")
    
    df = pd.read_csv(input_path)
    
    train_parts = []
    test_parts = []
    
    for i in range(0, len(df), block_size):
        block = df.iloc[i:i+block_size].copy()
        # Shuffle within block
        block = block.sample(frac=1, random_state=random_state).reset_index(drop=True)
        
        n_train = int(len(block) * train_ratio)
        
        train_parts.append(block.iloc[:n_train])
        test_parts.append(block.iloc[n_train:])
    
    train_df = pd.concat(train_parts, ignore_index=True)
    test_df = pd.concat(test_parts, ignore_index=True)
    
    print(f"Node {node} - Train rows: {len(train_df)}, Test rows: {len(test_df)}")
    
    # Save to normalized directory (these are the files your normalization code expects)
    train_out_path = os.path.join(output_dir, f"Node_{node}_train.csv")
    test_out_path = os.path.join(output_dir, f"Node_{node}_test.csv")
    
    train_df.to_csv(train_out_path, index=False)
    test_df.to_csv(test_out_path, index=False)

print("Done. All splits saved to:", output_dir)

Processing dataset/Node_A_final_synthetic_dataset_with_source.csv ...
Node A - Train rows: 50826, Test rows: 21801
Processing dataset/Node_B_final_synthetic_dataset_with_source.csv ...
Node B - Train rows: 50826, Test rows: 21801
Processing dataset/Node_C_final_synthetic_dataset_with_source.csv ...
Node C - Train rows: 50826, Test rows: 21801
Processing dataset/Node_D_final_synthetic_dataset_with_source.csv ...
Node D - Train rows: 50826, Test rows: 21801
Processing dataset/Node_E_final_synthetic_dataset_with_source.csv ...
Node E - Train rows: 50826, Test rows: 21801
Processing dataset/Node_F_final_synthetic_dataset_with_source.csv ...
Node F - Train rows: 50826, Test rows: 21801
Processing dataset/Node_G_final_synthetic_dataset_with_source.csv ...
Node G - Train rows: 50826, Test rows: 21801
Processing dataset/Node_H_final_synthetic_dataset_with_source.csv ...
Node H - Train rows: 50826, Test rows: 21801
Done. All splits saved to: dataset/normalized


In [39]:
#Cell 2
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# =============================================================================
# Configuration
# =============================================================================
nodes = ["A", "B", "C", "D", "E", "F", "G", "H"]
base_dir = "dataset/normalized"

block_size = 1689  # not used here, but kept for consistency if needed
random_state = 42
np.random.seed(random_state)

# Ensure output directory exists
os.makedirs(base_dir, exist_ok=True)

In [40]:
#Cell 3
# =============================================================================
# Normalize numeric features to [0, 1] using train statistics (per node)
# =============================================================================

for node in nodes:
    print(f"\n=== Processing Node {node} ===")
    
    # Load train and test splits for this node
    train_path = os.path.join(base_dir, f"Node_{node}_train.csv")
    test_path = os.path.join(base_dir, f"Node_{node}_test.csv")
    
    print(f"Loading {train_path} and {test_path} ...")
    
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    
    # Identify numeric columns (exclude non-numeric / categorical)
    numeric_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()
    
    # Optionally exclude any numeric ID-like columns if present
    # e.g. if you had an 'id' column:
    # numeric_cols = [c for c in numeric_cols if c not in ["id"]]
    
    print("Numeric columns to normalize:", numeric_cols)
    
    # Initialize scaler
    scaler = MinMaxScaler(feature_range=(0, 1))
    
    # Fit on train, transform train and test
    train_df_norm = train_df.copy()
    test_df_norm = test_df.copy()
    
    train_df_norm[numeric_cols] = scaler.fit_transform(train_df[numeric_cols])
    test_df_norm[numeric_cols] = scaler.transform(test_df[numeric_cols])
    
    # Verify normalization
    print("\n--- Normalization Verification ---")
    print(f"Train - Min: {train_df_norm[numeric_cols].min().min():.6f}, Max: {train_df_norm[numeric_cols].max().max():.6f}")
    print(f"Test  - Min: {test_df_norm[numeric_cols].min().min():.6f}, Max: {test_df_norm[numeric_cols].max().max():.6f}")
    
    # =============================================================================
    # Save normalized datasets
    # =============================================================================
    norm_train_path = os.path.join(base_dir, f"Node_{node}_train_normalized.csv")
    norm_test_path = os.path.join(base_dir, f"Node_{node}_test_normalized.csv")

    train_df_norm.to_csv(norm_train_path, index=False)
    test_df_norm.to_csv(norm_test_path, index=False)

    print(f"\nNormalized datasets saved for Node {node}:")
    print(f"  Train: {norm_train_path}")
    print(f"  Test:  {norm_test_path}")

    print("\nDone. All normalized datasets are in:", base_dir)


=== Processing Node A ===
Loading dataset/normalized/Node_A_train.csv and dataset/normalized/Node_A_test.csv ...
Numeric columns to normalize: ['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW']

--- Normalization Verification ---
Train - Min: 0.000000, Max: 1.000000
Test  - Min: 0.011761, Max: 1.029441

Normalized datasets saved for Node A:
  Train: dataset/normalized/Node_A_train_normalized.csv
  Test:  dataset/normalized/Node_A_test_normalized.csv

Done. All normalized datasets are in: dataset/normalized

=== Processing Node B ===
Loading dataset/normalized/Node_B_train.csv and dataset/normalized/Node_B_test.csv ...
Numeric columns to normalize: ['shunt_voltage', 'bus_voltage_V', 'current_mA', 'power_mW']

--- Normalization Verification ---
Train - Min: 0.000000, Max: 1.000000
Test  - Min: 0.003382, Max: 1.009261

Normalized datasets saved for Node B:
  Train: dataset/normalized/Node_B_train_normalized.csv
  Test:  dataset/normalized/Node_B_test_normalized.csv

Done. All n